# LLM–Brain Alignment — Surprisal vs Structure (Pereira2018)

Colab driver. Clones the repo, installs deps, runs the 5-stage pipeline, and plots
layer-depth curves + variance partitioning. See `README.md`, `LOG.md`, `TRACKER.md`.

**Core question:** how much LLM–brain alignment on Pereira2018 is *uniquely* explained by
hidden states beyond surprisal, and how does that depend on size / layer / instruction tuning.

## 1. Setup — clone + install

In [ ]:
# In Colab: clone the repo (replace with your remote) and install.
import os
REPO = 'LLM-BrainAlignment'
if not os.path.exists(REPO):
    !git clone https://github.com/<you>/{REPO}.git
%cd {REPO}
!pip -q install -e .[models,plot]

In [ ]:
# Optional: install the brain-score language harness for real Pereira2018.
# Heavy install — only run when you want real data.
# !pip -q install git+https://github.com/brain-score/language

## 2. Smoke test (synthetic, no models)
Confirms the full chain works before pulling weights or data.

In [ ]:
!python -m pytest tests/ -q
for stage in ['01_extract_activations','02_compute_surprisal','03_fit_encoding',
              '04_variance_partitioning','05_rsa']:
    !python scripts/{stage}.py --synthetic

## 3. Real run — one model
Set `MODEL` to a key from `config/default.yaml` (gpt2, pythia-160m, pythia-410m, opt-125m, ...).
Drop `--synthetic`. Requires the brain-score harness (cell above) for real Pereira2018.

In [ ]:
MODEL = 'gpt2'
!python scripts/01_extract_activations.py --model {MODEL}
!python scripts/02_compute_surprisal.py  --model {MODEL}
!python scripts/03_fit_encoding.py       --model {MODEL}
!python scripts/04_variance_partitioning.py --model {MODEL}
!python scripts/05_rsa.py                --model {MODEL}

## 4. Plots — layer-depth predictivity & variance partitioning

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

MODEL = 'gpt2'
d = Path('data/derived')/MODEL
enc = json.load(open(d/'encoding.json'))
vp  = json.load(open(d/'varpart.json'))

fig, ax = plt.subplots(1, 2, figsize=(12,4))
layers = sorted(int(l) for l in enc['per_layer'])
ax[0].plot(layers, [enc['per_layer'][str(l)]['mean_r'] for l in layers], 'o-')
ax[0].set(title=f'{MODEL}: layer-depth predictivity', xlabel='layer', ylabel='mean r')

vl = sorted(int(l) for l in vp['per_layer'])
uh = [vp['per_layer'][str(l)]['mean_unique_hidden'] for l in vl]
us = [vp['per_layer'][str(l)]['mean_unique_surprisal'] for l in vl]
sh = [vp['per_layer'][str(l)]['mean_shared'] for l in vl]
ax[1].stackplot(vl, uh, sh, us, labels=['unique hidden','shared','unique surprisal'])
ax[1].set(title=f'{MODEL}: variance partitioning', xlabel='layer', ylabel='R²')
ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

## 5. Model-family sweep
Run several models, then compare best-layer unique-hidden across size / instruction tuning.
Record each run in `TRACKER.md`.

In [ ]:
MODELS = ['gpt2','pythia-160m','pythia-410m','opt-125m']
for m in MODELS:
    for stage in ['01_extract_activations','02_compute_surprisal',
                  '03_fit_encoding','04_variance_partitioning','05_rsa']:
        !python scripts/{stage}.py --model {m}